# Serve Unlimited-OCR on Colab GPU
Runtime -> Change runtime type -> **GPU (T4)**, then Run All.
The last cell prints the public URL to put in `~/.config/market-secrets/credentials.env` as `UNLIMITED_OCR_URL=...` on the Mac.

In [ ]:
!nvidia-smi -L
!pip -q install vllm pymupdf

In [ ]:
# Start vLLM with the OpenAI-compatible API in the background.
# --limit-mm-per-prompt lets the client send many pages per request
# (the model's whole point: dozens of pages in one forward pass).
import subprocess, time, requests
proc = subprocess.Popen([
    'vllm', 'serve', 'baidu/Unlimited-OCR',
    '--trust-remote-code', '--port', '8000',
    '--limit-mm-per-prompt', '{"image": 32}',
    '--max-model-len', '32768',
])
for _ in range(180):  # first run downloads weights; be patient
    time.sleep(10)
    try:
        if requests.get('http://localhost:8000/v1/models', timeout=5).ok:
            print('vLLM is up'); break
    except Exception:
        pass

In [ ]:
# Expose it with a Cloudflare quick tunnel (no account needed).
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
import subprocess, re, time
tun = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
t0 = time.time()
while time.time() - t0 < 60 and url is None:
    line = tun.stdout.readline()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m: url = m.group(0)
print('UNLIMITED_OCR_URL =', url)

Keep this notebook running while you use the endpoint. Colab free tier disconnects after a few hours — rerun when needed.